ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(16, 12), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('Distribution of All Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, color='steelblue', edgecolors='none', s=10)
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, color='tomato', edgecolors='none', s=10)
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop('Cluster', axis=1) if 'Cluster' in df.columns else df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Inertia vs K')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='green')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs K')
plt.xticks(range(2, 11))
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': [round(s, 4) for s in silhouette_scores]
})
silhouette_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# I chose K=4 based on both the elbow method and business reasoning.
# The elbow curve shows a noticeable bend around K=3 to K=4, and K=4 gives
# a good balance between cluster separation and meaningful customer segments.

kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = kmeans_final.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pca_components, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df['Cluster'].values

plt.figure(figsize=(9, 6))
colors = ['steelblue', 'tomato', 'green', 'orange']
for cluster in sorted(pca_df['Cluster'].unique()):
    subset = pca_df[pca_df['Cluster'] == cluster]
    plt.scatter(subset['PC1'], subset['PC2'], s=10, alpha=0.4,
                label=f'Cluster {cluster}', color=colors[cluster])

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('K-Means Clusters Visualized with PCA (2D)')
plt.legend()
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

2. Why did we remove the `CUST_ID` column?

3. Which columns had missing values?

4. How did you handle the missing values?

5. Why is scaling important before applying K-Means?

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

7. Based on the cluster summary table, describe each customer segment in your own words.

8. Which cluster may represent high-value customers?

9. Which cluster may represent customers who rely more on cash advance?

10. How can a company use these clusters for marketing strategy?

---

**Answers:**

**1. Why is this an unsupervised learning problem?**

This is an unsupervised learning problem because the dataset doesn't have any predefined labels or categories for the customers. We don't already know which group each customer belongs to — the goal is to discover those groups from the data itself. There's no target column telling us "this customer is in segment A," so we have to let the algorithm figure out the patterns on its own.

**2. Why did we remove the `CUST_ID` column?**

The `CUST_ID` column is just an identifier — it's a unique code assigned to each customer and doesn't carry any information about their behavior. If we kept it, it would confuse the clustering algorithm because it would try to find patterns in random ID numbers, which don't mean anything. Removing it makes sure the model only looks at actual behavioral features.

**3. Which columns had missing values?**

Two columns had missing values: `CREDIT_LIMIT` had 1 missing value, and `MINIMUM_PAYMENTS` had 313 missing values.

**4. How did you handle the missing values?**

I used mean imputation — I filled each missing value with the mean of that column. This is a simple and reasonable approach for numerical data. It avoids losing rows and keeps the distribution roughly the same.

**5. Why is scaling important before applying K-Means?**

K-Means works by calculating the distance between data points and cluster centroids. If one feature has much larger values than another (for example, `CREDIT_LIMIT` can be in the thousands while `PURCHASES_FREQUENCY` is between 0 and 1), the algorithm will be dominated by the larger-scale feature. Scaling makes sure every feature contributes equally to the distance calculation.

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

I chose K=4. Looking at the elbow curve, the decrease in inertia starts to slow down noticeably around K=3 to K=4, which suggests that adding more clusters after K=4 doesn't give much improvement. The silhouette score was highest at K=3 (around 0.25), but K=4 still gave a reasonable score and produces more meaningful segments for a business use case. Four segments — like high spenders, cash advance users, average users, and low activity customers — makes more practical sense than just three.

**7. Based on the cluster summary table, describe each customer segment in your own words.**

Looking at the cluster means:

- **Cluster 0** — These are moderate-balance customers with decent purchases and relatively low cash advance usage. They seem to be regular shoppers who use their cards for everyday purchases. Balance is around $895 and purchases around $1,236.

- **Cluster 1** — This is the high-value group. They have the highest purchases (~$7,682), highest credit limit (~$9,697), and highest payments (~$7,289). These are active, high-spending customers who also pay a lot back.

- **Cluster 2** — These customers have a very high balance (~$4,602) and very high cash advance (~$4,521), but relatively low purchases (~$502). They rely heavily on cash advances rather than purchases. This might indicate financial stress or a specific pattern of using the card as a cash source.

- **Cluster 3** — These are low-activity customers. They have low balance, low purchases, and low payments. They probably don't use their credit cards very much.

**8. Which cluster may represent high-value customers?**

Cluster 1 most likely represents high-value customers. They have the highest purchases, highest credit limit, and highest payments. These are the customers the company probably wants to retain and reward.

**9. Which cluster may represent customers who rely more on cash advance?**

Cluster 2 is clearly the cash advance group. Their average cash advance is around $4,521, which is much higher than any other cluster, while their purchases are very low. They seem to be using the credit card mainly to withdraw cash rather than make purchases.

**10. How can a company use these clusters for marketing strategy?**

- **Cluster 1 (High-value spenders):** Offer premium rewards, higher credit limits, and loyalty programs to keep them engaged.
- **Cluster 0 (Regular shoppers):** Target them with cashback offers or purchase-based promotions to increase spending.
- **Cluster 2 (Cash advance users):** These customers may be in financial difficulty. The company could offer balance transfer options or financial planning tools. They could also be targeted carefully with low-interest offers.
- **Cluster 3 (Low activity):** These customers are not using their cards much. The company could send re-engagement campaigns, special offers, or reminders to encourage more card usage.

Overall, customer segmentation helps the company avoid a one-size-fits-all approach and instead create targeted, more effective campaigns for each group.